# ML Baseline — Prix au m² (LightGBM, walk-forward)

Modèle prédictif traditionnel sur `rpt_features_commune_mois`.

- **Grain** : commune × mois
- **Cible** : `prix_m2_med`
- **Features** : structurelles + lags 1/3/6/12m + rolling 3/6/12m
- **Validation** : walk-forward (train N années, test année N+1)
- **Métriques** : MAE, RMSE, MAPE globaux + par département

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import clickhouse_connect
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)),
    username=os.getenv('CLICKHOUSE_USER', 'admin'),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
)
print('ClickHouse connecté')

## 1. Chargement des données

In [ ]:
query = """
SELECT
    code_commune,
    code_departement,
    annee,
    mois,
    -- cible
    prix_m2_med,
    -- volume
    nb_transactions,
    -- caractéristiques du parc
    pct_appt,
    pct_maison,
    surface_moy,
    nb_pieces_moy,
    -- DPE
    pct_passoires_thermiques,
    conso_ep_moy,
    -- construction
    nb_commences,
    taux_concretisation,
    -- logement social
    nb_entrees_ls,
    -- marché neuf
    prix_m2_neuf_moy,
    delai_ecoulement_moy,
    -- lags & rolling
    prix_m2_lag_1m,
    prix_m2_lag_3m,
    prix_m2_lag_6m,
    prix_m2_lag_12m,
    prix_m2_roll3m,
    prix_m2_roll6m,
    prix_m2_roll12m,
    evol_1m_pct,
    evol_12m_pct,
    -- encodage cyclique
    mois_sin,
    mois_cos
FROM db_ai_house.rpt_features_commune_mois
WHERE prix_m2_med > 0
  AND prix_m2_lag_12m IS NOT NULL
ORDER BY annee, mois, code_commune
"""

df = client.query_df(query)
df['periode'] = df['annee'] * 100 + df['mois']
print(f'{len(df):,} lignes | {df["code_commune"].nunique():,} communes | {df["annee"].min()}–{df["annee"].max()}')
df.head(3)

## 2. Configuration features / cible

In [ ]:
TARGET = 'prix_m2_med'

CAT_FEATURES = ['code_departement']

NUM_FEATURES = [
    'nb_transactions',
    'pct_appt', 'pct_maison', 'surface_moy', 'nb_pieces_moy',
    'pct_passoires_thermiques', 'conso_ep_moy',
    'nb_commences', 'taux_concretisation',
    'nb_entrees_ls',
    'prix_m2_neuf_moy', 'delai_ecoulement_moy',
    'prix_m2_lag_1m', 'prix_m2_lag_3m', 'prix_m2_lag_6m', 'prix_m2_lag_12m',
    'prix_m2_roll3m', 'prix_m2_roll6m', 'prix_m2_roll12m',
    'evol_1m_pct', 'evol_12m_pct',
    'mois_sin', 'mois_cos',
]

ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

for col in CAT_FEATURES:
    df[col] = df[col].astype('category')

print(f'{len(ALL_FEATURES)} features | cible : {TARGET}')

## 3. Walk-forward cross-validation

In [ ]:
def mape(y_true, y_pred):
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

PARAMS = {
    'objective':        'regression',
    'metric':           'mae',
    'n_estimators':     1500,
    'learning_rate':    0.02,
    'num_leaves':       63,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'n_jobs':           -1,
    'verbose':          -1,
}

annees = sorted(df['annee'].unique())
# Walk-forward : train jusqu'à N-1, test sur N
folds = [(annees[:i], annees[i]) for i in range(2, len(annees))]

results = []
models  = []

for train_years, test_year in folds:
    train = df[df['annee'].isin(train_years)]
    test  = df[df['annee'] == test_year]

    X_train, y_train = train[ALL_FEATURES], train[TARGET]
    X_test,  y_test  = test[ALL_FEATURES],  test[TARGET]

    model = lgb.LGBMRegressor(**PARAMS)
    model.fit(
        X_train, y_train,
        categorical_feature=CAT_FEATURES,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        eval_set=[(X_test, y_test)],
    )

    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mape_ = mape(y_test.values, preds)

    print(f'Fold test={test_year} | train={train_years[0]}–{train_years[-1]} | MAE={mae:.0f} €/m² | RMSE={rmse:.0f} | MAPE={mape_:.1f}%')
    results.append({'test_year': test_year, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape_})
    models.append(model)

results_df = pd.DataFrame(results)
print('\n--- Moyenne walk-forward ---')
print(results_df[['MAE','RMSE','MAPE']].mean().round(1))

## 4. Métriques par département (dernier fold)

In [ ]:
last_model = models[-1]
last_year  = folds[-1][1]
test_last  = df[df['annee'] == last_year].copy()
test_last['pred'] = last_model.predict(test_last[ALL_FEATURES])
test_last['err']  = (test_last['pred'] - test_last[TARGET]).abs()

dept_metrics = (
    test_last.groupby('code_departement')
    .agg(
        MAE  =('err', 'mean'),
        n    =(TARGET, 'count'),
        prix_reel_med=(TARGET, 'median'),
    )
    .sort_values('MAE', ascending=False)
    .reset_index()
)

print(f'Top 10 départements les plus difficiles à prédire (test {last_year}) :')
dept_metrics.head(10)

## 5. Importance des features

In [ ]:
imp = pd.DataFrame({
    'feature':    last_model.feature_name_,
    'importance': last_model.feature_importances_,
}).sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(imp['feature'], imp['importance'], color='steelblue')
ax.set_xlabel('Importance LightGBM (gain)')
ax.set_title(f'Top 20 features — LightGBM walk-forward (test {last_year})')
plt.tight_layout()
plt.show()

## 6. Prédiction vs réel — distribution des erreurs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter pred vs réel
ax = axes[0]
sample = test_last.sample(min(5000, len(test_last)), random_state=42)
ax.scatter(sample[TARGET], sample['pred'], alpha=0.2, s=5, color='steelblue')
lim = (sample[TARGET].min(), sample[TARGET].max())
ax.plot(lim, lim, 'r--', lw=1)
ax.set_xlabel('Prix réel (€/m²)')
ax.set_ylabel('Prix prédit (€/m²)')
ax.set_title('Prédit vs Réel')

# Distribution erreurs absolues
ax = axes[1]
test_last['err'].clip(0, 1000).hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(test_last['err'].median(), color='red', linestyle='--', label=f'Médiane = {test_last["err"].median():.0f} €/m²')
ax.set_xlabel('Erreur absolue (€/m²)')
ax.set_ylabel('Nombre de communes×mois')
ax.set_title('Distribution des erreurs')
ax.legend()

plt.tight_layout()
plt.show()

print(f"MAE final ({last_year}) : {test_last['err'].mean():.0f} €/m²")
print(f"MAPE final ({last_year}) : {mape(test_last[TARGET].values, test_last['pred'].values):.1f}%")

## 7. Résumé walk-forward

In [ ]:
print('=== RÉSULTATS WALK-FORWARD ===')
print(results_df.to_string(index=False))
print()
print('--- Moyenne ---')
print(f"MAE  : {results_df['MAE'].mean():.0f} €/m²")
print(f"RMSE : {results_df['RMSE'].mean():.0f} €/m²")
print(f"MAPE : {results_df['MAPE'].mean():.1f}%")